## IDP Phase: Extraction
In this notebook, we will see how to perform extraction in BDA using the BDA boto3 APIs

### Step-1: Setup environment
* Setup the environemnt. Restart the kernel after this step

In [ ]:
%pip install --no-warn-conflicts "boto3>=1.37.6" itables==2.2.4 PyPDF2==3.0.1 --upgrade -qq

In [1]:
%load_ext autoreload
%autoreload 1

### Step-2: Initialize Parameters
* Before we get to the part where we invoke BDA with our sample artifacts, let's setup some parameters and configuration that will be used throughout this notebook.

In [ ]:
import boto3
import json
from IPython.display import JSON, IFrame
import sagemaker
import pandas as pd
from utils import display_functions, helper_functions
from pathlib import Path
import os

session = sagemaker.Session()
default_bucket = session.default_bucket()
current_region = boto3.session.Session().region_name

sts_client = boto3.client('sts')
account_number = sts_client.get_caller_identity()['Account']
# Define bucket and folder paths
bucket_name = f"idp-workshop-{account_number}-us-west-2"
output_folder = "BDA-output"  # Write extracted data

# Initialize Bedrock Data Automation client
bda_client = boto3.client('bedrock-data-automation')
bda_runtime_client = boto3.client('bedrock-data-automation-runtime')
s3_client = boto3.client('s3')

bda_s3_input_location = f's3://{default_bucket}/bda/input'
bda_s3_output_location = f's3://{default_bucket}/bda/output'
print(bda_s3_input_location)

### Step-3: Prepare sample document
* Let upload the sample document from local folder **patient-records/inference-records** to S3


In [ ]:
local_download_path = '../patient-records/inference-records'
local_file_name = 'Simple_Medical_Chart_Christopher_Daniels_inf_1.pdf'
local_file_path = os.path.join(local_download_path, local_file_name)

document_s3_uri = f'{bda_s3_input_location}/{local_file_name}'

target_s3_bucket, target_s3_key =  helper_functions.get_bucket_and_key(document_s3_uri)
s3_client.upload_file(local_file_path, target_s3_bucket, target_s3_key)

print(f"Sample file local path: {local_file_path}")
print(f"Uploaded file to S3: {target_s3_key}")
print(f"document_s3_uri: {document_s3_uri}")

### Step-4: View Sample Document

In [ ]:
IFrame(local_file_path, width=600, height=400)

### Step-4: Get Project ARN
* go to the BDA project console and copy the Project ARN for **idp-bda-workshop-project**
* Also verify that the project is in good state

### Step-5: Async Extraction for Simple Medical Chart
* Invoke Data Automation Async API for extraction of **simple_medical_chart_inf.pdf**
* This step returns the invocation ARN

In [ ]:
response = bda_runtime_client.invoke_data_automation_async(
    inputConfiguration={
        's3Uri': document_s3_uri
    },
    outputConfiguration={
        's3Uri': bda_s3_output_location
    },
    dataAutomationConfiguration={
        'dataAutomationProjectArn': project_arn,
        'stage': 'LIVE'
    }, 
    dataAutomationProfileArn = f'arn:aws:bedrock:{current_region}:{account_id}:data-automation-profile/us.data-automation-v1'
)

invocationArn = response['invocationArn']
print(f'Invoked data automation job with invocation arn {invocationArn}')

### Step-6: Poll for Invocation Status
* Since invocation is an async step, we will poll for status to be completed
* The invocation job status moves from `Created` to `InProgress` and finally to `Success` when the job completes successfully
* If the job encounters an error the final status is either `ServiceError` or `ClientError` with error details
* Run this step couple of time till the status is `Success`

In [ ]:
status_response = helper_functions.wait_for_completion(
            client=bda_client,
            get_status_function=bda_runtime_client.get_data_automation_status,
            status_kwargs={'invocationArn': invocationArn},
            completion_states=['Success'],
            error_states=['ClientError', 'ServiceError'],
            status_path_in_response='status',
            max_iterations=15,
            delay=30
)
if status_response['status'] == 'Success':
    job_metadata_s3_location = status_response['outputConfiguration']['s3Uri']
else:
    raise Exception(f'Invocation Job Error, error_type={status_response["error_type"]},error_message={status_response["error_message"]}')

### Step-7: Validate Classification
* Retrieve the job metadata. The Job metadata contains the S3 uri's for the standard output,custom output and the status of custom output. The custom output status could be either of `MATCH` or `NO_MATCH`.
* `MATCH` indicates BDA was able to find a matching blueprint for the specific segment from the list of blueprint we associated with the project.
* Verify the **Document Class** is **Simple Patient Medical Chart**

In [ ]:
job_metadata = json.loads(helper_functions.read_s3_object(job_metadata_s3_location))

job_metadata_table = pd.DataFrame(job_metadata['output_metadata'][0]['segment_metadata']).fillna('')
job_metadata_table.index.name='Segment Index'
job_metadata_json = JSON(job_metadata, root="job_metadata", expanded=True)
# Display the widget
display_functions.display_multiple(
    [display_functions.get_view(job_metadata_table), display_functions.get_view(job_metadata_json)], 
    ["Table View", "Raw JSON"])

### Step-7: Explore the Custom Insights

### View Segments and Matched Blueprints
As we can see in the `job metadata`, BDA creates a segment section each for each individual document that it has identified in the file. Each segment section has details on the matched blueprint and the results of the extraction. For each segment, BDA also outputs the page indices (one or more) from the original file.

We can now get the custom output corresponding to each segment and look at the insights that BDA custom output produces.

In [16]:
asset_id = 0
segments_metadata = next(item["segment_metadata"]
                                for item in job_metadata["output_metadata"] 
                                if item['asset_id'] == asset_id)

standard_outputs = [
    json.loads(helper_functions.read_s3_object(segment_metadata.get('standard_output_path')))for segment_metadata in segments_metadata]
custom_outputs = [json.loads(helper_functions.read_s3_object(segment_metadata.get('custom_output_path'))) if segment_metadata.get('custom_output_status') == 'MATCH' else None for segment_metadata in segments_metadata]


### View Custom output summary

In [ ]:
custom_outputs_json = JSON(custom_outputs, root="custom_outputs", expanded=False)
custom_outputs_table = pd.DataFrame(helper_functions.get_summaries(custom_outputs)).fillna('')

display_functions.display_multiple(
    [
        display_functions.get_view(custom_outputs_table.style.hide(axis='index')),
        display_functions.get_view(custom_outputs_json)
    ], 
    ["Table View", "Raw JSON"])

### Explore Document Insights using Standard and Custom output

In [ ]:
views=[]
titles=[]
# Use the function
for custom_output, standard_output in zip(custom_outputs, standard_outputs):
    if custom_output:
        result = helper_functions.transform_custom_output(custom_output['inference_result'], custom_output['explainability_info'][0])
        document_image_uris = [page.get('asset_metadata',{}).get('rectified_image') for page in standard_output.get('pages',[])]
        views += [display_functions.segment_view(document_image_uris=document_image_uris,
                   inference_result=result)]
        titles += [custom_output.get('matched_blueprint', {}).get('name', None)]
display_functions.display_multiple(views, titles)